In [3]:
# ════════════════════════════════════════════════════════════════
# Imports
# ════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', 'tqdm', '-q'])

import os, glob, json, math, pickle
import xml.etree.ElementTree as ET
from collections import defaultdict
import numpy as np
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

PIE_PATH        = '/content/drive/MyDrive/PIE'
LANDMARK_PATH   = f'{PIE_PATH}/landmarks'
ANNOTATION_PATH = f'{PIE_PATH}/annotations'
ATTR_PATH       = f'{PIE_PATH}/annotations_attributes'
OUTPUT_PATH     = f'{PIE_PATH}/processed_features/features'
os.makedirs(OUTPUT_PATH, exist_ok=True)

SEQ_LEN     = 30
SEQ_STEP    = 15
WINDOW_SIZE = 10
FPS         = 30

LABEL_NAMES = {0:'Waiting', 1:'Hesitant', 2:'Committed',
               3:'Distracted', 4:'Aggressive', 5:'Jaywalk'}

NOSE=0; L_EAR=7; R_EAR=8
L_SHOULDER=11; R_SHOULDER=12
L_ELBOW=13;    R_ELBOW=14
L_HIP=23;      R_HIP=24
L_KNEE=25;     R_KNEE=26
L_ANKLE=27;    R_ANKLE=28
L_HEEL=29;     R_HEEL=30
L_FOOT_IDX=31; R_FOOT_IDX=32

FEATURE_NAMES = [
    'current_speed','avg_speed','max_speed','acceleration','deceleration',
    'speed_variance','step_frequency','left_step_length','right_step_length',
    'pause_between_steps','upper_body_angle','lower_body_angle','head_angle',
    'head_turn_frequency','shoulder_angle','hip_angle','foot_angle_left',
    'foot_angle_right','forward_lean','lateral_lean','body_orientation',
    'body_orientation_change','pause_duration','hesitation_cycles',
    'total_hesitation_time','distance_to_curb','distance_change_rate',
    'temporal_movement_probability','looks_left','looks_right',
    'direction_changes','traffic_light_state','vehicle_distance',
    'crosswalk_presence','road_width','pedestrian_density'
]

SETS = ['set01','set02','set03','set04','set05','set06']

print('Configuration loaded.')
print(f'  Output path : {OUTPUT_PATH}')
print(f'  Features    : {len(FEATURE_NAMES)}')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Configuration loaded.
  Output path : /content/drive/MyDrive/PIE/processed_features/features
  Features    : 36


In [4]:

# ════════════════════════════════════════════════════════════════
# Geometry Helpers
# ════════════════════════════════════════════════════════════════
def load_landmarks(npy_path):
    return np.load(npy_path).reshape(33, 3)

def midpoint(a, b):
    return (np.array(a) + np.array(b)) / 2.0

def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

def angle_between(p1, vertex, p2):
    v1 = np.array(p1) - np.array(vertex)
    v2 = np.array(p2) - np.array(vertex)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2), -1.0, 1.0))))

def vertical_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return float(np.degrees(np.arctan2(dx, -dy)))

def horizontal_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    if abs(dx) < 1e-6:
        return 90.0
    return float(np.degrees(np.arctan(dy / dx)))

print('Geometry helpers defined.')



Geometry helpers defined.


In [5]:

# ════════════════════════════════════════════════════════════════
# XML Parsers
# ════════════════════════════════════════════════════════════════
def parse_annotation_xml(xml_path):
    frame_data = defaultdict(lambda: {
        'pedestrians': [], 'traffic_lights': [], 'crosswalks': [], 'vehicles': []
    })
    try:
        root = ET.parse(xml_path).getroot()
    except Exception as e:
        print(f'  Warning: cannot parse {xml_path}: {e}')
        return frame_data

    for track in root.findall('track'):
        label = track.attrib.get('label', '')
        for box in track.findall('box'):
            if box.attrib.get('outside', '0') == '1':
                continue
            fid  = int(box.attrib['frame'])
            bbox = [float(box.attrib.get(k, 0)) for k in ('xtl','ytl','xbr','ybr')]
            occ_raw = box.attrib.get('occluded', '0')
            occ     = int(occ_raw)
            attrs   = {a.attrib['name']: (a.text or '').strip()
                       for a in box.findall('attribute')}

            if label == 'pedestrian':
                ped_id   = attrs.get('id', '')
                occ_text = attrs.get('occlusion', 'none')
                occ_num  = {'none': 0, 'part': 1, 'full': 2}.get(occ_text, occ)
                frame_data[fid]['pedestrians'].append({
                    'id':        ped_id,
                    'bbox':      bbox,
                    'occlusion': occ_num,
                    'action':    attrs.get('action',  'standing'),
                    'gesture':   attrs.get('gesture', 'none'),
                    'look':      attrs.get('look',    'not-looking'),
                    'cross':     attrs.get('cross',   'not-crossing')
                })
            elif label == 'traffic_light':
                attrs2 = {a.attrib['name']: (a.text or '').strip()
                          for a in box.findall('attribute')}
                frame_data[fid]['traffic_lights'].append({
                    'state': attrs2.get('state', 'none'),
                    'bbox':  bbox
                })
            elif label == 'crosswalk':
                frame_data[fid]['crosswalks'].append({'bbox': bbox})
            elif label == 'vehicle':
                attrs2 = {a.attrib['name']: (a.text or '').strip()
                          for a in box.findall('attribute')}
                frame_data[fid]['vehicles'].append({
                    'bbox': bbox,
                    'type': attrs2.get('type', 'car')
                })
    return frame_data


def parse_attributes_xml(attr_path):
    ped_attrs = {}
    if not attr_path or not os.path.exists(attr_path):
        return ped_attrs
    try:
        root = ET.parse(attr_path).getroot()
    except:
        return ped_attrs
    for ped in root.findall('.//pedestrian'):
        pid = ped.attrib.get('id', '')
        ped_attrs[pid] = {
            'intention_prob': float(ped.attrib.get('intention_prob', 0.5)),
            'crossing':       int(ped.attrib.get('crossing', 0)),
            'num_lanes':      int(ped.attrib.get('num_lanes', 2)),
            'signalized':     ped.attrib.get('signalized', 'n/a'),
            'crossing_point': int(ped.attrib.get('crossing_point', -1)),
            'critical_point': int(ped.attrib.get('critical_point', -1)),
        }
    return ped_attrs

print('XML parsers defined.')



XML parsers defined.


In [6]:

# ════════════════════════════════════════════════════════════════
# Context Helpers
# ════════════════════════════════════════════════════════════════
TL_MAP = {'red': 0.0, 'yellow': 0.5, 'green': 1.0, 'none': -1.0}

def bbox_center(bbox):
    x1,y1,x2,y2 = bbox
    return ((x1+x2)/2, (y1+y2)/2)

def bbox_iou(b1, b2):
    ix1=max(b1[0],b2[0]); iy1=max(b1[1],b2[1])
    ix2=min(b1[2],b2[2]); iy2=min(b1[3],b2[3])
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    if inter == 0: return 0.0
    a1=(b1[2]-b1[0])*(b1[3]-b1[1])
    a2=(b2[2]-b2[0])*(b2[3]-b2[1])
    return inter/(a1+a2-inter+1e-6)

def ctx_traffic_light(frame_info, ped_bbox):
    lights = frame_info.get('traffic_lights', [])
    if not lights: return -1.0
    pcx,pcy = bbox_center(ped_bbox)
    best_d, best_s = float('inf'), 'none'
    for tl in lights:
        cx,cy = bbox_center(tl['bbox'])
        d = math.hypot(cx-pcx, cy-pcy)
        if d < best_d: best_d, best_s = d, tl.get('state','none')
    return TL_MAP.get(best_s, -1.0)

def ctx_vehicle_distance(frame_info, ped_bbox):
    vehicles = frame_info.get('vehicles', [])
    if not vehicles: return 1.0
    pcx,pcy = bbox_center(ped_bbox)
    min_d = min(math.hypot(*[c-p for c,p in zip(bbox_center(v['bbox']),(pcx,pcy))])
                for v in vehicles)
    return float(np.clip(min_d/1920.0, 0.0, 1.0))

def ctx_crosswalk(frame_info, ped_bbox):
    return 1.0 if any(bbox_iou(ped_bbox, cw['bbox']) > 0.01
                      for cw in frame_info.get('crosswalks',[])) else 0.0

def ctx_road_width(num_lanes):
    return float(np.clip(num_lanes/6.0, 0.0, 1.0))

def ctx_ped_density(frame_info, ped_id):
    count = sum(1 for p in frame_info.get('pedestrians',[]) if p['id'] != ped_id)
    return float(np.clip(count/10.0, 0.0, 1.0))

print('Context helpers defined.')





Context helpers defined.


In [7]:


# ════════════════════════════════════════════════════════════════
# Features
# ════════════════════════════════════════════════════════════════
def compute_36_features(lm_window, frame_info, ped_entry, num_lanes, history):
    F      = np.zeros(36, dtype=np.float32)
    lm     = lm_window[-1]
    bbox   = ped_entry['bbox']
    ped_id = ped_entry['id']

    bw = max(bbox[2]-bbox[0], 1.0)
    bh = max(bbox[3]-bbox[1], 1.0)

    def px(idx, lm_arr=None):
        arr = lm_arr if lm_arr is not None else lm
        return np.array([arr[idx,0]*bw, arr[idx,1]*bh])

    speeds = []
    for i in range(1, len(lm_window)):
        h_p = midpoint(px(L_HIP,lm_window[i-1]), px(R_HIP,lm_window[i-1]))
        h_c = midpoint(px(L_HIP,lm_window[i]),   px(R_HIP,lm_window[i]))
        speeds.append(euclidean(h_c, h_p))
    if not speeds: speeds = [0.0]
    spd = np.array(speeds)

    F[0] = spd[-1]
    F[1] = float(spd.mean())
    F[2] = float(spd.max())
    F[3] = max(spd[-1]-spd[-2], 0.0) if len(spd)>=2 else 0.0
    F[4] = max(spd[-2]-spd[-1], 0.0) if len(spd)>=2 else 0.0
    F[5] = float(spd.var())

    STEP_TH = 2.0
    step_ev = 0; still_cnt = 0
    for i in range(1, len(lm_window)):
        la  = px(L_ANKLE,lm_window[i]);   la_p = px(L_ANKLE,lm_window[i-1])
        ra  = px(R_ANKLE,lm_window[i]);   ra_p = px(R_ANKLE,lm_window[i-1])
        if euclidean(la,la_p) > STEP_TH:  step_ev += 1
        if euclidean(la,la_p) < STEP_TH and euclidean(ra,ra_p) < STEP_TH: still_cnt += 1

    n_w  = max(len(lm_window)-1, 1)
    F[6] = float(np.clip(step_ev/n_w,   0.0, 1.0))

    la_c = px(L_ANKLE); ra_c = px(R_ANKLE)
    la_p = px(L_ANKLE,lm_window[-2]) if len(lm_window)>=2 else la_c
    ra_p = px(R_ANKLE,lm_window[-2]) if len(lm_window)>=2 else ra_c
    F[7] = euclidean(la_c,la_p)
    F[8] = euclidean(ra_c,ra_p)
    F[9] = float(np.clip(still_cnt/n_w, 0.0, 1.0))

    sl=px(L_SHOULDER); sr=px(R_SHOULDER)
    hl=px(L_HIP);      hr=px(R_HIP)
    kl=px(L_KNEE);     kr=px(R_KNEE)
    al=px(L_ANKLE);    ar=px(R_ANKLE)
    fl=px(L_FOOT_IDX); fr=px(R_FOOT_IDX)
    ns=px(NOSE)

    s_mid=midpoint(sl,sr); h_mid=midpoint(hl,hr)
    k_mid=midpoint(kl,kr); a_mid=midpoint(al,ar)

    F[10] = vertical_angle(h_mid,s_mid)
    F[11] = angle_between(h_mid,k_mid,a_mid)
    F[12] = vertical_angle(s_mid,ns)

    nose_xs = [px(NOSE,lm_window[i])[0] for i in range(len(lm_window))]
    scx_avg = np.mean([(px(L_SHOULDER,lm_window[i])[0]+px(R_SHOULDER,lm_window[i])[0])/2
                        for i in range(len(lm_window))])
    sides = [1 if x > scx_avg else -1 for x in nose_xs]
    turns = sum(1 for i in range(1,len(sides)) if sides[i]!=sides[i-1])
    F[13] = float(np.clip(turns/n_w, 0.0, 1.0))

    F[14] = horizontal_angle(sl,sr)
    F[15] = horizontal_angle(hl,hr)
    F[16] = horizontal_angle(al,fl)
    F[17] = horizontal_angle(ar,fr)
    F[18] = abs(F[10])
    F[19] = abs(s_mid[0]-h_mid[0])
    F[20] = horizontal_angle(sl,sr)

    orients = [horizontal_angle(px(L_SHOULDER,lm_window[i]),
                                px(R_SHOULDER,lm_window[i]))
               for i in range(len(lm_window))]
    oc = [abs(orients[i]-orients[i-1]) for i in range(1,len(orients))]
    F[21] = float(np.mean(oc)) if oc else 0.0

    SPEED_TH = 1.0
    is_still = F[0] < SPEED_TH
    moving   = not is_still

    history['still']  = history.get('still',0)  + (1 if is_still else 0)
    history['htime']  = history.get('htime',0)  + (1 if not moving else 0)
    if moving != history.get('was_moving', False):
        history['hcycles'] = history.get('hcycles',0) + 1
    history['was_moving'] = moving

    n_hist = max(history.get('total_frames',1), 1)
    history['total_frames'] = history.get('total_frames',0) + 1

    F[22] = float(np.clip(history['still']/n_hist,        0.0, 1.0))
    F[23] = float(np.clip(history.get('hcycles',0)/10.0, 0.0, 1.0))
    F[24] = float(np.clip(history['htime']/n_hist,        0.0, 1.0))
    F[25] = float(np.clip(max(lm[L_ANKLE,1],lm[R_ANKLE,1]), 0.0, 1.0))

    prev_ay = max(lm_window[-2][L_ANKLE,1],lm_window[-2][R_ANKLE,1]) \
              if len(lm_window)>=2 else F[25]
    F[26] = float(F[25]-prev_ay)

    p_spd = float(np.clip(F[1]/10.0, 0.0, 1.0))
    p_var = float(np.clip(F[5]/50.0, 0.0, 1.0))
    p_stp = float(np.clip(F[6],      0.0, 1.0))
    F[27] = float(np.clip((p_spd+p_var+p_stp)/3.0, 0.0, 1.0))

    nose_x_n = lm[NOSE,0]
    scx_n    = (lm[L_SHOULDER,0]+lm[R_SHOULDER,0])/2.0
    F[28] = 1.0 if nose_x_n < scx_n else 0.0
    F[29] = 1.0 if nose_x_n > scx_n else 0.0

    deltas = [orients[i]-orients[i-1] for i in range(1,len(orients))]
    dc = sum(1 for i in range(1,len(deltas)) if (deltas[i-1]>0)!=(deltas[i]>0))
    F[30] = float(np.clip(dc/10.0, 0.0, 1.0))

    F[31] = ctx_traffic_light(frame_info,bbox)
    F[32] = ctx_vehicle_distance(frame_info,bbox)
    F[33] = ctx_crosswalk(frame_info,bbox)
    F[34] = ctx_road_width(num_lanes)
    F[35] = ctx_ped_density(frame_info,ped_id)

    return F

print('compute_36_features() defined.')



compute_36_features() defined.


In [8]:



all_raw_features = []   # (N, 36)
all_raw_gestures = []   # gesture string per frame
all_raw_crosses  = []   # cross string per frame
all_raw_meta     = []   # (set_id, vid, ped_id, fid)
all_raw_attrs    = []   # intention_prob per frame

for set_id in SETS:
    lm_set_dir   = f'{LANDMARK_PATH}/{set_id}'
    ann_set_dir  = f'{ANNOTATION_PATH}/{set_id}'
    attr_set_dir = f'{ATTR_PATH}/{set_id}'

    if not os.path.isdir(lm_set_dir):
        print(f'Skipping {set_id} — no landmark folder'); continue

    ann_files = sorted(glob.glob(f'{ann_set_dir}/*_annt.xml'))
    if not ann_files:
        ann_files = sorted(glob.glob(f'{ann_set_dir}/*.xml'))

    print(f'\n{"="*55}')
    print(f'{set_id}: {len(ann_files)} annotation files found')
    print(f'{"="*55}')

    for ann_file in ann_files:
        base       = os.path.basename(ann_file)
        vid_folder = base.replace(f'{set_id}_','').replace('_annt.xml','').replace('.xml','')
        lm_vid_dir = f'{lm_set_dir}/{vid_folder}'

        if not os.path.isdir(lm_vid_dir):
            print(f'  Skip {base} — no landmark folder'); continue

        attr_file = f'{attr_set_dir}/{base.replace("_annt.xml","_attributes.xml")}'
        if not os.path.exists(attr_file):
            candidates = glob.glob(f'{attr_set_dir}/*{vid_folder}*')
            attr_file  = candidates[0] if candidates else None

        ped_attrs_map = parse_attributes_xml(attr_file)
        frame_data    = parse_annotation_xml(ann_file)

        lm_files = sorted(glob.glob(f'{lm_vid_dir}/frame_*.npy'))
        if not lm_files:
            print(f'    No .npy files in {lm_vid_dir}'); continue

        lm_map = {}
        for lf in lm_files:
            fid = int(os.path.basename(lf).replace('frame_','').replace('.npy',''))
            lm_map[fid] = lf

        ped_track_frames = defaultdict(list)
        for fid in sorted(lm_map.keys()):
            if fid not in frame_data: continue
            for ped in frame_data[fid]['pedestrians']:
                if ped['occlusion'] < 2 and ped['id'] != '':
                    ped_track_frames[ped['id']].append(fid)

        print(f'  Parsing: {base}')
        print(f'    {len(lm_map)} frames | {len(ped_track_frames)} tracks')

        for ped_id, track_frames in tqdm(ped_track_frames.items(),
                                          desc=f'    {vid_folder}', leave=False):
            track_frames = sorted(track_frames)
            if len(track_frames) < SEQ_LEN:
                continue

            attrs     = ped_attrs_map.get(ped_id, {'intention_prob':0.5,'num_lanes':2})
            num_lanes = attrs.get('num_lanes', 2)
            lm_window = []
            history   = {}

            for fid in track_frames:
                if fid not in lm_map: continue
                try:
                    lm = load_landmarks(lm_map[fid])
                except:
                    continue

                # ── Landmark zero check ──────────────────────
                if np.all(lm == 0):
                    continue

                lm_window.append(lm)
                if len(lm_window) > WINDOW_SIZE:
                    lm_window.pop(0)
                if len(lm_window) < 2:
                    continue

                frame_info = frame_data.get(fid, {
                    'pedestrians':[],'traffic_lights':[],'crosswalks':[],'vehicles':[]
                })
                ped_entry = next((p for p in frame_info['pedestrians']
                                  if p['id']==ped_id), None)
                if ped_entry is None: continue

                # ── Bbox sanity check ────────────────────────
                x1,y1,x2,y2 = ped_entry['bbox']
                if (x2-x1) < 1 or (y2-y1) < 1:
                    continue

                # ── STEP 1: Compute & save  features ──────
                feats = compute_36_features(
                    lm_window, frame_info, ped_entry, num_lanes, history)

                # Save
                all_raw_features.append(feats.copy())
                all_raw_gestures.append(ped_entry.get('gesture', 'none'))
                all_raw_crosses.append(ped_entry.get('cross',   'not-crossing'))
                all_raw_meta.append((set_id, vid_folder, ped_id, fid))
                all_raw_attrs.append({
                    'intention_prob': attrs.get('intention_prob', 0.5),
                    'num_lanes':      num_lanes,
                    'frame_info':     frame_info,
                    'ped_entry':      ped_entry,
                })

# ── Save features  ────────────────────────────
X_raw = np.array(all_raw_features, dtype=np.float32)
np.save(f'{OUTPUT_PATH}/X_raw.npy',
        X_raw)
np.save(f'{OUTPUT_PATH}/raw_gestures.npy',
        np.array(all_raw_gestures, dtype=object))
np.save(f'{OUTPUT_PATH}/raw_crosses.npy',
        np.array(all_raw_crosses,  dtype=object))

print(f'\n{"="*55}')
print(f'STEP 1 COMPLETE — Raw features extracted ')
print(f'  Total frames     : {len(X_raw):,}')
print(f'  X_raw   : {X_raw.shape}')
print(f'  Saved to         : {OUTPUT_PATH}')
print(f'{"="*55}')




set01: 4 annotation files found
  Parsing: video_0001_annt.xml
    86 frames | 17 tracks


  Parsing: video_0002_annt.xml
    1031 frames | 35 tracks


  Parsing: video_0003_annt.xml
    903 frames | 44 tracks


  Parsing: video_0004_annt.xml
    133 frames | 8 tracks



set02: 3 annotation files found
  Parsing: video_0001_annt.xml
    860 frames | 33 tracks


...

  Parsing: video_0006_annt.xml
    7 frames | 9 tracks


  Parsing: video_0007_annt.xml
    873 frames | 15 tracks


  Parsing: video_0008_annt.xml
    180 frames | 13 tracks


  Parsing: video_0009_annt.xml
    353 frames | 29 tracks



STEP 1 COMPLETE — Raw features extracted 
  Total frames     : 73,743
  X_raw   : (73743, 36)
  Saved to         : /content/drive/MyDrive/PIE/processed_features/features


In [11]:

# ════════════════════════════════════════════════════════════════
# = Label Assignment=
# ════════════════════════════════════════════════════════════════
def assign_label(ped_entry, attrs, frame_info, speed_var):
    intention = attrs.get('intention_prob', 0.5)
    gesture   = ped_entry.get('gesture', 'none')
    look      = ped_entry.get('look',    'not-looking')
    cross     = ped_entry.get('cross',   'not-crossing')
    crosswalk = ctx_crosswalk(frame_info, ped_entry['bbox'])

    if cross == 'crossing' and crosswalk == 0.0:                return 5  # Jaywalk
    if gesture in {'hand_yield','hand_rightofway','hand_ack'} \
       or speed_var > 100.0:                                     return 4  # Aggressive
    if intention > 0.66 and cross == 'crossing':                return 2  # Committed
    if 0.33 <= intention <= 0.66:                                return 1  # Hesitant
    if intention < 0.33 and look == 'not-looking':              return 3  # Distracted
    return 0                                                               # Waiting

print('assign_label() defined.')


assign_label() defined.


In [12]:


# ════════════════════════════════════════════════════════════════
# Assign Labels to Raw Features
# ════════════════════════════════════════════════════════════════
print('Assigning labels to raw features...')

all_labels = []
for i in range(len(all_raw_features)):
    attrs_i     = all_raw_attrs[i]
    ped_entry_i = attrs_i['ped_entry']
    frame_info_i= attrs_i['frame_info']
    speed_var_i = float(all_raw_features[i][F_SPEED_VARIANCE])

    label = assign_label(
        ped_entry_i,
        attrs_i,
        frame_info_i,
        speed_var=speed_var_i
    )
    all_labels.append(label)

X = np.array(all_raw_features, dtype=np.float32)
y = np.array(all_labels,       dtype=np.int32)

bad = int(np.isnan(X).sum()) + int(np.isinf(X).sum())
if bad:
    print(f'Fixing {bad} NaN/Inf → 0')
    X = np.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0)
else:
    print('No NaN/Inf ')

print(f'\nX shape : {X.shape}')
print(f'y shape : {y.shape}')

print('\nClass distribution (per frame):')
for lid, lname in LABEL_NAMES.items():
    n   = int(np.sum(y==lid))
    pct = n/len(y)*100
    print(f'  {lid} {lname:<15} {n:>8,}  ({pct:5.1f}%)  {" "*int(pct/2)}')

np.save(f'{OUTPUT_PATH}/X_features.npy', X)
np.save(f'{OUTPUT_PATH}/y_labels.npy',   y)
np.save(f'{OUTPUT_PATH}/meta.npy', np.array(all_raw_meta, dtype=object))
print(f'\nSaved X_features.npy  {X.shape}')
print(f'Saved y_labels.npy    {y.shape}')
print(f'Saved meta.npy        {len(all_raw_meta):,} entries')




Assigning labels to raw features...
No NaN/Inf 

X shape : (73743, 36)
y shape : (73743,)

Class distribution (per frame):
  0 Waiting           33,069  ( 44.8%)                        
  1 Hesitant           4,479  (  6.1%)     
  2 Committed         13,064  ( 17.7%)          
  3 Distracted         5,358  (  7.3%)     
  4 Aggressive        11,745  ( 15.9%)         
  5 Jaywalk            6,028  (  8.2%)      

Saved X_features.npy  (73743, 36)
Saved y_labels.npy    (73743,)
Saved meta.npy        73,743 entries


In [13]:

# ════════════════════════════════════════════════════════════════
# Fit Scaler
# ════════════════════════════════════════════════════════════════
N      = len(X)
scaler = StandardScaler()
scaler.fit(X[:int(N*0.70)])
X_scaled = scaler.transform(X).astype(np.float32)

with open(f'{OUTPUT_PATH}/feature_scaler.pkl','wb') as f:
    pickle.dump(scaler, f)

np.save(f'{OUTPUT_PATH}/X_features_scaled.npy', X_scaled)
print('Scaler fitted and saved.')
print(f'Scaler mean[:5] : {scaler.mean_[:5].round(3)}')
print(f'Scaler std[:5]  : {scaler.scale_[:5].round(3)}')


Scaler fitted and saved.
Scaler mean[:5] : [ 6.01   6.124 18.119  2.69   2.705]
Scaler std[:5]  : [12.788  7.936 25.059  9.332  9.514]


In [14]:

# ════════════════════════════════════════════════════════════════
# Track-Aware Split
# ════════════════════════════════════════════════════════════════
track_frames_map = defaultdict(list)
for i, (sid, vid, pid, fid) in enumerate(all_raw_meta):
    track_frames_map[(sid, vid, pid)].append(i)

track_ids    = list(track_frames_map.keys())
track_labels = []
for tid in track_ids:
    idxs = track_frames_map[tid]
    track_labels.append(int(np.bincount([y[i] for i in idxs]).argmax()))

track_ids    = np.array(track_ids,    dtype=object)
track_labels = np.array(track_labels, dtype=np.int32)

tr_tracks, tmp_tracks, tr_tl, tmp_tl = train_test_split(
    track_ids, track_labels,
    test_size=0.30, random_state=42, stratify=track_labels)

va_tracks, te_tracks = train_test_split(
    tmp_tracks, tmp_tl,
    test_size=0.50, random_state=42, stratify=tmp_tl)[:2]

tr_set = set(map(tuple, tr_tracks))
va_set = set(map(tuple, va_tracks))
te_set = set(map(tuple, te_tracks))

assert len(tr_set & va_set) == 0
assert len(tr_set & te_set) == 0
assert len(va_set & te_set) == 0
print('── Track Leakage Check ──')
print(f'  Train/Val  overlap :    0 ')
print(f'  Train/Test overlap :    0 ')
print(f'  Val/Test   overlap :    0 ')
print(f'\n  Train tracks : {len(tr_set)}')
print(f'  Val   tracks : {len(va_set)}')
print(f'  Test  tracks : {len(te_set)}')


── Track Leakage Check ──
  Train/Val  overlap :    0 
  Train/Test overlap :    0 
  Val/Test   overlap :    0 

  Train tracks : 491
  Val   tracks : 105
  Test  tracks : 106
